# Device-resident autograd — Phase 1 parity (CPU vs GPU)

`pure/dtensor.hpp`: tensor data/grad in a `thrust::device_vector`; ops via thrust +
`bk::gemm`; reverse-mode backward on the selected backend. Same source → CPU or GPU.

**Runtime → GPU (T4)**, Run all. Both builds must print the same `L` and checksums
(`dL/dX`, `dL/dW`) to ~1e-4, and `grad-check OK`.


In [ ]:
!nvidia-smi -L
!nvcc --version | tail -1


### Clone `thrust-backend`


In [ ]:
%cd /content
!rm -rf yolov8_cpp
!git clone -q --branch thrust-backend https://github.com/yomei-o/yolov8_cpp.git
%cd /content/yolov8_cpp


### GPU (nvcc) build + run


In [ ]:
# GPU build: -arch=native (real GPU) + -DUSE_CUDA (bk:: device path, matches thrust device_vector).
!nvcc -x cu -O2 -std=c++17 --extended-lambda -arch=native -DUSE_CUDA pure/dtest.cpp -o dtest_gpu
!./dtest_gpu; echo "exit=$?"


### CPU (g++, same source) build + run


In [ ]:
!g++ -O2 -std=c++17 -I/usr/local/cuda/include -I/usr/local/cuda/include/cccl \
     -DTHRUST_DEVICE_SYSTEM=THRUST_DEVICE_SYSTEM_CPP pure/dtest.cpp -o dtest_cpu
!./dtest_cpu


---
**Expected** (matches the local MSVC CPU run):
`L = 3.001538`, `sum(dL/dX)=1.732878`, `sum(dL/dW)=8.266924`, `grad-check OK` on both.
Tiny CPU/GPU differences (~1e-4) are float reduction-order, not bugs.
